### Pada bagian ini, akan digunakan **TestsetGenerator** dari RAGAS untuk membuat data uji sintetis untuk model retrieval

In [1]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
    ],
}

In [2]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [7]:
docs_splits = text_splitter.split_documents(pages)

In [8]:
docs_splits[0]

Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/', 'title': '1. Satuan Kredit Semester (SKS) – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='1. Satuan Kredit Semester (SKS) – Pedoman Akademik STT-NF\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nADMINISTRASI\n\n\n1. Daftar Ulang\n2. Administrasi Keuangan\n3. BAAK\n4. UPT Perpustakaan\n\n\n\n\n\nKELULUSAN\n\n\n1. Syarat Kelulusan\n2. Yudisium\n3. Syarat Pengambilan SKL, Transkrip, dan Ijazah\n4. Wisuda\n\n\n\n\n\nMBKM\n\n\n1. Pedoman MBKM\n2. Administrasi\n3. Konversi SKS\n\n\n\n\n\nORGANISASI KEMAHASISWAAN\n\n\n1. Tentang Organisasi Kemahasiswaan\n2. Aturan\n3. Fasilitas Mahasiswa\n4. Soft Skill\n5. Mars STT Terpadu Nurul Fikri\n6. Hymne STT Terpadu Nurul Fikri\n\n\n\n\n\nPENUNJANG AKADEMIK\n\n\n1. Penelitian Dan PKM\n2. Laboratorium\n3. Perpustakaan\n4. Pengembangan Diri Mahasiswa\n5. Unit Pendukung Kemahasi

In [9]:
from langchain_community.retrievers import BM25Retriever

retriever = BM25Retriever.from_documents(docs_splits)
retriever.k = 3

Pada bagian atas adalah persiapan data rule atau pedomana. Data diambil dari Website STTNF, kemudian di featch dengan langchain dan di chunking sebelum diuji ke RAGAS

In [10]:
from langchain_community.chat_models import ChatOllama

llm_model = ChatOllama(
    model='mistral-openorca:7b-q4_0',
    temperature=0
)

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [12]:
setup_and_retrieval = RunnableParallel(
    {
        "context":retriever,
        "question":RunnablePassthrough()
    }
)

In [13]:
template = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
)

In [14]:
rag_chain = setup_and_retrieval | template | llm_model | StrOutputParser()

In [15]:
from datasets import Dataset

/Users/a/Programming/Langchain-Project/my-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
questions = [
    "Berapa jumlah minimal SKS yang harus diselesaikan mahasiswa STT Terpadu Nurul Fikri untuk bisa lulus?",
    "Berapa batas minimum IPK agar mahasiswa dinyatakan lulus?",
    "Berapa batas maksimal masa studi untuk program Sarjana?"
]

ground_truths = [
    "Mahasiswa harus menyelesaikan minimal 148 SKS sesuai ketentuan program studi.",
    "Mahasiswa harus memiliki IPK minimal 2.00.",
    "Masa studi tidak boleh melebihi 7 tahun atau 14 semester."
]

In [28]:
answers = []
contexts = []

for query in questions:
    answers.append(rag_chain.invoke(query))
    contexts.append([docs.page_content for docs in retriever._get_relevant_documents(query, run_manager="CallbackManagerForRetrieverRun")])


data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(data)

In [29]:
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas import evaluate

In [31]:
import os

os.environ['OPENAI_API_KEY'] = 'dummy_key'

In [32]:
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        context_recall, 
        context_precision, 
        answer_relevancy
    ],
)

df = result.to_pandas()

Evaluating:   8%|▊         | 1/12 [00:00<00:04,  2.66it/s]Exception raised in Job[2]: AuthenticationError(Error code: 401 - {'error': {'message': 'Incorrect API key provided: dummy_key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}})
Exception raised in Job[1]: AuthenticationError(Error code: 401 - {'error': {'message': 'Incorrect API key provided: dummy_key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}})
Exception raised in Job[3]: AuthenticationError(Error code: 401 - {'error': {'message': 'Incorrect API key provided: dummy_key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}})
Exception raised in Job[10]: AuthenticationError(Error code: 401 - {'error': {'message': 'Incorr

In [33]:
df

,question,answer,contexts,ground_truth,faithfulness,context_recall,context_precision,answer_relevancy
0,Berapa jumlah minimal SKS yang harus diselesai...,Berapa jumlah minimal SKS yang harus diselesa...,[Jadi nilai IP berkisar antara 0 – 4\nEvaluasi...,Mahasiswa harus menyelesaikan minimal 148 SKS ...,NaN,NaN,NaN,NaN
1,Berapa batas minimum IPK agar mahasiswa dinyat...,Berapa batas minimum IPK agar mahasiswa dinya...,[Masa Studi\nKetentuan masa studi adalah sebag...,Mahasiswa harus memiliki IPK minimal 2.00.,NaN,NaN,NaN,NaN
2,Berapa batas maksimal masa studi untuk program...,Program sarjana harus diselesaikan dalam wakt...,[Masa Studi\nKetentuan masa studi adalah sebag...,Masa studi tidak boleh melebihi 7 tahun atau 1...,NaN,NaN,NaN,NaN
